# Option portfolio validation

Build a book of one European option and one Asian option, then inspect a single structured report before any simulation.

`PortfolioEvolver.run_monte_carlo` does not start unless `validate_report()` is ok. A deterministic forward curve is not a stochastic scenario generator, so this notebook stops at that gate.

Build first, and shut the kernel down before rebuilding:

`cmake --build --preset debug --target sjd_python`

In [1]:
import math
import sys
from datetime import date
from pathlib import Path

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _notebooks = _candidate / "python" / "notebooks"
    if (_notebooks / "sjd_setup.py").is_file():
        if str(_notebooks) not in sys.path:
            sys.path.insert(0, str(_notebooks))
        break
    if (_candidate / "sjd_setup.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise RuntimeError("Could not find python/notebooks/sjd_setup.py from " + str(_cwd))

import sjd_setup

sjd = sjd_setup.load_sjd()

def show_report(title, report):
    print(f"\n{title}")
    print("OK:", report.ok)
    if not report.issues:
        print("No issues")
        return
    print(f"{'Code':<42} {'Path'}")
    for issue in report.issues:
        print(f"{issue.code:<42} {'.'.join(issue.path)}")

anchor = date(2026, 9, 16)
settlement = date(2026, 12, 16)
european = sjd.EuropeanOption(
    maturity=settlement,
    settlement=settlement,
    strike=17.60,
    type=sjd.PutCall.CALL,
    volume=1_000_000.0,
    premium_payment_date=anchor,
    premium_amount=0.0,
    buy_sell=sjd.BuySell.BUY,
)
asian = sjd.AsianOption(
    averaging_dates=[anchor, settlement],
    settlement=settlement,
    strike=17.40,
    type=sjd.PutCall.PUT,
    volume=500_000.0,
    premium_payment_date=anchor,
    premium_amount=0.0,
    buy_sell=sjd.BuySell.BUY,
)
portfolio = sjd.Portfolio([european, asian])
print(sjd.__file__)


E:\Code\sjd\python\sjd_runtime\sjd.pyd


## Incomplete market

An empty rates environment produces one report for the whole book. Each issue names the contract index.

In [2]:
empty = sjd.RatesEnvironment(anchor)
incomplete = portfolio.validate_report(empty)
show_report("Book against an empty market", incomplete)
assert not incomplete.ok
assert any(
    issue.path[:3] == ("portfolio", "contracts[0]", "european_option")
    and issue.code == "MARKET_VOLATILITY_RATE_SOURCE_MISSING"
    for issue in incomplete.issues
)
assert any(
    issue.path[:3] == ("portfolio", "contracts[1]", "asian_option")
    and issue.code == "MARKET_FIXING_RATE_SOURCE_MISSING"
    for issue in incomplete.issues
)


Book against an empty market
OK: False
Code                                       Path
RATES_ENVIRONMENT_NO_RATES                 portfolio.rates_environment
RATES_ENVIRONMENT_NO_RATES                 portfolio.contracts[0].european_option.rates_environment
MARKET_DISCOUNT_RATE_SOURCE_MISSING        portfolio.contracts[0].european_option.rates_environment.discount_rate_source
MARKET_FORWARD_RATE_SOURCE_MISSING         portfolio.contracts[0].european_option.rates_environment.forward_rate_source
MARKET_VOLATILITY_RATE_SOURCE_MISSING      portfolio.contracts[0].european_option.rates_environment.volatility_rate_source
RATES_ENVIRONMENT_NO_RATES                 portfolio.contracts[1].asian_option.rates_environment
MARKET_DISCOUNT_RATE_SOURCE_MISSING        portfolio.contracts[1].asian_option.rates_environment.discount_rate_source
MARKET_FORWARD_RATE_SOURCE_MISSING         portfolio.contracts[1].asian_option.rates_environment.forward_rate_source
MARKET_VOLATILITY_RATE_SOURCE_MISSING      po

## Book is ready to value

Discount, forward, volatility, and fixing curves that cover the settlement and averaging dates clear the portfolio report. This market can price the contracts. It still cannot drive a Monte Carlo path, because the forward does not roll stochastically.

In [3]:
pillars = [anchor, settlement]
environment = sjd.RatesEnvironment(anchor)
environment.set_drs(sjd.DiscountCurve(anchor, pillars, [1.0, 0.98]))
environment.set_frs(sjd.ForwardCurve(anchor, pillars, [17.50, 17.80]))
environment.set_vrs(sjd.VolatilityCurve(anchor, pillars, [0.20, 0.20]))
environment.set_fixing_rs(sjd.DailyFixingCurve([anchor], [17.50]))

ready = portfolio.validate_report(environment)
show_report("Book against a complete deterministic market", ready)
assert ready.ok

generator = sjd.SingleFactorEconomicScenarioGenerator(environment)
evolver = sjd.PortfolioEvolver(portfolio, generator, [anchor, settlement])
blocked = evolver.validate_report()
show_report("Evolver preflight", blocked)
assert not blocked.ok
assert any(issue.code == "PATH_GENERATOR_FORWARD_NOT_STOCHASTIC" for issue in blocked.issues)
evolver.run_monte_carlo(10)
print("run_monte_carlo returned without starting paths")


Book against a complete deterministic market
OK: True
No issues

Evolver preflight
OK: False
Code                                       Path
PATH_GENERATOR_FORWARD_NOT_STOCHASTIC      portfolio_evolver.path_generator.forward_rate_source
run_monte_carlo returned without starting paths


## Where the paths are

A successful spot path and Monte Carlo, including margin and regulatory capital, are in [portfolio_paths.ipynb](portfolio_paths.ipynb). That notebook gives the same forward a `ForwardRoll.FORWARD_VOLATILITY` roll.
